In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets
import timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Preprocessing obrazów
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Automatyczne wczytanie z folderów
full_dataset = datasets.ImageFolder(root='dataset/crop', transform=transform)
classes = full_dataset.classes

# Podział na Train (75%), Valid (18%), Test (7%)
train_size = int(0.75 * len(full_dataset))
val_size = int(0.18 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_ds, val_ds, test_ds = random_split(full_dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

Using device: cuda


In [2]:
model = timm.create_model('efficientnet_b3', pretrained=True, num_classes=len(classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)

c:\Moje\Repositories\aircraft-classification\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Kubat\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 49.3MB            

model.safetensors: downloading bytes:           |  0.00B            

c:\Moje\Repositories\aircraft-classification\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Kubat\.cache\huggingface\hub\models--timm--efficientnet_b3.ra2_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [4]:
from tqdm import tqdm

# Ustawienie num_workers uwalnia pełną moc GPU na lokalnej maszynie
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)

epochs = 15
best_val_loss = float('inf')

for epoch in range(epochs):
    # --- FAZA TRENINGU ---
    model.train()
    train_loss, train_correct, total_train = 0.0, 0, 0
    
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{epochs:02d} [Train]", leave=False)
    for images, labels in train_loop:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()
        total_train += labels.size(0)
        
        # Aktualizacja statusu na żywo co batch
        train_loop.set_postfix(loss=loss.item())

    epoch_train_loss = train_loss / total_train
    epoch_train_acc = train_correct / total_train

    # --- FAZA WALIDACJI ---
    model.eval()
    val_loss, val_correct, total_val = 0.0, 0, 0
    val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1:02d}/{epochs:02d} [Val]", leave=False)
    
    with torch.no_grad():
        for images, labels in val_loop:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            total_val += labels.size(0)

    epoch_val_loss = val_loss / total_val
    epoch_val_acc = val_correct / total_val

    # Wypisanie czytelnego podsumowania epoki
    print(f"Epoch {epoch+1:02d}/{epochs:02d} | "
          f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc*100:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc*100:.2f}%")

    # Checkpointing (Zapis najlepszego modelu)
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"  --> Saved new best checkpoint (Val Loss: {epoch_val_loss:.4f})")

Epoch 01/15 | Train Loss: 1.3171 Acc: 65.13% | Val Loss: 0.9811 Acc: 73.04%
  --> Saved new best checkpoint (Val Loss: 0.9811)


Epoch 02/15 | Train Loss: 0.7933 Acc: 77.73% | Val Loss: 0.9049 Acc: 76.88%
  --> Saved new best checkpoint (Val Loss: 0.9049)


Epoch 03/15 | Train Loss: 0.5820 Acc: 83.56% | Val Loss: 0.7640 Acc: 79.68%
  --> Saved new best checkpoint (Val Loss: 0.7640)


Epoch 04/15 | Train Loss: 0.4580 Acc: 86.72% | Val Loss: 0.8119 Acc: 80.55%


Epoch 05/15 | Train Loss: 0.3574 Acc: 89.47% | Val Loss: 0.8081 Acc: 80.64%


Epoch 06/15 | Train Loss: 0.3047 Acc: 90.73% | Val Loss: 0.8130 Acc: 80.84%


... (2 outputs left out)

Epoch 09/15 | Train Loss: 0.2180 Acc: 93.29% | Val Loss: 0.7991 Acc: 82.45%


Epoch 10/15 | Train Loss: 0.1954 Acc: 93.90% | Val Loss: 0.8505 Acc: 81.17%


Epoch 11/15 | Train Loss: 0.1852 Acc: 94.24% | Val Loss: 0.8343 Acc: 81.90%


KeyboardInterrupt: 

In [5]:
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

C:\Users\Kubat\AppData\Local\Temp\ipykernel_17388\3866026890.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model.pth'))


Test Accuracy: 78.81%
